# Lab 6 — FSM-Controlled Agent
**Pattern: FINITE STATE MACHINE** — from HTML Section 04

---

## What is a State Machine?

Every agent framework — LangGraph, CrewAI, Bedrock Agents — implements a state machine under the hood.  
Most of the time, that state is **hidden**. You can't see it. When something goes wrong, you're debugging blind.

This lab makes the state machine **explicit**. Every state is named. Every transition is validated before it fires. Every move is logged.

---

## Real-world analogy

Think of a traffic light controller.

- It can only be in **GREEN**, **AMBER**, or **RED**
- It **cannot jump from GREEN to RED directly** — AMBER is required
- The rules about valid transitions are the **guards**

Your agent is the same:
- It cannot move from `PLANNING` to `DONE` without passing through `ACTING` and `OBSERVING`
- If it tries, the guard fires immediately — you catch the bug in tests, not in production

---

## The five states

| State | What the agent is doing |
|-------|------------------------|
| `PLANNING` | Calling the LLM — deciding what to do next |
| `CALLING` | Executing a tool the model requested |
| `OBSERVING` | Reading the tool result, processing it |
| `REFLECTING` | Deciding: is the goal met, or do I need more work? |
| `DONE` | Clean exit — task complete |
| `ERROR` | Budget exhausted or unrecoverable failure |

---

## What you will learn
1. How to name and track agent state explicitly
2. How guards prevent illegal transitions
3. How a budget limit is just a state machine guard
4. How to produce a full audit trail of every state change
5. Why every framework is secretly doing this

## Step 1 — Setup

In [ ]:
import anthropic
from dotenv import load_dotenv
from enum import Enum

load_dotenv()
client = anthropic.Anthropic()
print("✓ Client ready")

## Step 2 — State Definitions

We define the states as an `Enum`. This means:
- States are **named constants** — no magic strings
- Typos raise an error immediately at the `AgentState.TYPO` line — not silently at runtime

In [ ]:
class AgentState(Enum):
    PLANNING   = "PLANNING"    # LLM is deciding what to do next
    CALLING    = "CALLING"     # Executing a tool call
    OBSERVING  = "OBSERVING"   # Reading the tool result
    REFLECTING = "REFLECTING"  # Evaluating whether the goal is met
    DONE       = "DONE"        # Goal achieved — clean exit
    ERROR      = "ERROR"       # Budget exhausted or unrecoverable failure

print("States defined:")
for state in AgentState:
    print(f"  {state.name}")

## Step 3 — Valid Transitions Table

This is the **allowed moves** table. Think of it like a map of roads.  
The agent can only travel along roads that exist on the map.  
Any other move raises an error **before** the transition happens.

In [ ]:
VALID_TRANSITIONS = {
    AgentState.PLANNING:   {AgentState.CALLING, AgentState.DONE},
    AgentState.CALLING:    {AgentState.OBSERVING, AgentState.ERROR},
    AgentState.OBSERVING:  {AgentState.REFLECTING, AgentState.ERROR},
    AgentState.REFLECTING: {AgentState.PLANNING, AgentState.DONE, AgentState.ERROR},
    AgentState.DONE:       set(),   # terminal — no exits
    AgentState.ERROR:      set(),   # terminal — no exits
}

# Visual check
print("Valid transitions:")
for state, targets in VALID_TRANSITIONS.items():
    arrows = " | ".join([t.value for t in targets]) if targets else "NONE (terminal)"
    print(f"  {state.value:12} → {arrows}")

## Step 4 — The FSM Agent Class

The agent wraps the state machine. Three key methods:

- `transition()` — validates and executes a state change, raises error if illegal
- `budget_guard()` — fires every turn, forces ERROR if max_turns exceeded  
- `run()` — the main loop, state transitions happen explicitly at each decision point

In [ ]:
class FSMAgent:
    def __init__(self, tools: list, tool_executor, max_turns: int = 10):
        self.tools         = tools
        self.tool_executor = tool_executor
        self.max_turns     = max_turns
        self.state         = AgentState.PLANNING
        self.turn_count    = 0
        self.messages      = []
        self.state_log     = []  # full audit trail

    def transition(self, new_state: AgentState, reason: str = ""):
        """
        Validates the transition BEFORE executing it.
        If the move is not in VALID_TRANSITIONS, raises ValueError immediately.
        This is the guard — it fires before any state change happens.
        """
        allowed = VALID_TRANSITIONS[self.state]
        if new_state not in allowed:
            raise ValueError(
                f"ILLEGAL TRANSITION: {self.state.value} → {new_state.value}. "
                f"Allowed from {self.state.value}: {[s.value for s in allowed]}"
            )
        old = self.state
        self.state = new_state
        entry = f"[Turn {self.turn_count}] {old.value} → {new_state.value}"
        if reason:
            entry += f"  ({reason})"
        self.state_log.append(entry)
        print(f"  [FSM] {entry}")

    def budget_guard(self) -> bool:
        """
        Fires EVERY turn — before anything else.
        No agent runs forever. This is the circuit breaker.
        """
        if self.turn_count >= self.max_turns:
            self.state = AgentState.ERROR
            self.state_log.append(f"[Turn {self.turn_count}] BUDGET GUARD → ERROR (max={self.max_turns})")
            print(f"  [FSM] ⚠ BUDGET GUARD FIRED: max_turns={self.max_turns} reached")
            return True
        return False

    def run(self, task: str, system_prompt: str) -> str:
        print(f"\n── Task: {task}")
        print(f"  [FSM] Starting state: {self.state.value}")
        self.messages = [{"role": "user", "content": task}]

        while self.state not in (AgentState.DONE, AgentState.ERROR):
            self.turn_count += 1

            # Budget guard runs first — every turn, no exceptions
            if self.budget_guard():
                break

            # ── PLANNING: call the model ──────────────────────────────────
            if self.state == AgentState.PLANNING:
                response = client.messages.create(
                    model="claude-sonnet-4-6",
                    max_tokens=1024,
                    system=system_prompt,
                    tools=self.tools,
                    messages=self.messages
                )
                self.messages.append({"role": "assistant", "content": response.content})

                if response.stop_reason == "tool_use":
                    self.transition(AgentState.CALLING, "model requested tool")
                elif response.stop_reason == "end_turn":
                    self.transition(AgentState.REFLECTING, "model finished thinking")
                else:
                    self.transition(AgentState.ERROR, f"unexpected stop_reason: {response.stop_reason}")
                self._pending_response = response

            # ── CALLING: execute tool(s) ──────────────────────────────────
            elif self.state == AgentState.CALLING:
                response = self._pending_response
                tool_results = []
                for block in response.content:
                    if block.type == "tool_use":
                        print(f"  [TOOL] {block.name}({block.input})")
                        try:
                            result = self.tool_executor(block.name, block.input)
                            tool_results.append({
                                "type": "tool_result",
                                "tool_use_id": block.id,
                                "content": result
                            })
                        except Exception as e:
                            tool_results.append({
                                "type": "tool_result",
                                "tool_use_id": block.id,
                                "content": f"Tool error: {str(e)}",
                                "is_error": True
                            })
                self.messages.append({"role": "user", "content": tool_results})
                self.transition(AgentState.OBSERVING, f"{len(tool_results)} result(s) collected")

            # ── OBSERVING: model reads tool results ───────────────────────
            elif self.state == AgentState.OBSERVING:
                response = client.messages.create(
                    model="claude-sonnet-4-6",
                    max_tokens=1024,
                    system=system_prompt,
                    tools=self.tools,
                    messages=self.messages
                )
                self.messages.append({"role": "assistant", "content": response.content})
                self._pending_response = response
                self.transition(AgentState.REFLECTING, "tool results processed")

            # ── REFLECTING: is the goal met? ──────────────────────────────
            elif self.state == AgentState.REFLECTING:
                response = self._pending_response
                if response.stop_reason == "end_turn":
                    self.transition(AgentState.DONE, "goal achieved")
                elif response.stop_reason == "tool_use":
                    self.transition(AgentState.PLANNING, "more tools needed")
                else:
                    self.transition(AgentState.ERROR, f"unexpected stop: {response.stop_reason}")

        # Extract final answer from message history
        final_text = ""
        for msg in reversed(self.messages):
            if msg["role"] == "assistant":
                content = msg["content"]
                if isinstance(content, list):
                    for block in content:
                        if hasattr(block, "text") and block.text:
                            final_text = block.text
                            break
                elif isinstance(content, str):
                    final_text = content
                if final_text:
                    break
        return final_text

print("✓ FSMAgent class defined")

## Step 5 — Tools

A simple calculator toolset. Intentionally simple — we want to watch the FSM transitions, not debug tool logic.

In [ ]:
TOOLS = [
    {
        "name": "calculate",
        "description": "Evaluate a mathematical expression. E.g. '42 * 365' or '(100 - 20) / 4'.",
        "input_schema": {
            "type": "object",
            "properties": {
                "expression": {"type": "string"}
            },
            "required": ["expression"]
        }
    },
    {
        "name": "convert_currency",
        "description": "Convert between currencies: USD, GBP, EUR, JPY, INR.",
        "input_schema": {
            "type": "object",
            "properties": {
                "amount":        {"type": "number"},
                "from_currency": {"type": "string"},
                "to_currency":   {"type": "string"}
            },
            "required": ["amount", "from_currency", "to_currency"]
        }
    }
]

RATES = {"USD": 1.0, "GBP": 0.79, "EUR": 0.92, "JPY": 149.5, "INR": 83.2}

def tool_executor(name: str, inputs: dict) -> str:
    if name == "calculate":
        expr = inputs["expression"]
        allowed = set("0123456789+-*/()., ")
        if not all(c in allowed for c in expr):
            return "Error: disallowed characters in expression"
        return f"Result: {eval(expr)}"
    elif name == "convert_currency":
        amount = inputs["amount"]
        frm = inputs["from_currency"].upper()
        to  = inputs["to_currency"].upper()
        if frm not in RATES or to not in RATES:
            return f"Unknown currency. Supported: {list(RATES.keys())}"
        converted = (amount / RATES[frm]) * RATES[to]
        return f"{amount} {frm} = {converted:.2f} {to}"
    return f"Unknown tool: {name}"

print("✓ Tools defined")

## Step 6 — Task 1: Single Tool Call

A simple calculation. Watch the FSM move through:  
`PLANNING → CALLING → OBSERVING → REFLECTING → DONE`

In [ ]:
SYSTEM = "You are a financial calculation assistant. Use tools for all calculations. Show your working."

agent1 = FSMAgent(tools=TOOLS, tool_executor=tool_executor, max_turns=8)
result1 = agent1.run(
    task="What is the monthly cost if an annual contract is worth $127,500?",
    system_prompt=SYSTEM
)
print(f"\n── Answer: {result1}")
print(f"\n── Final state: {agent1.state.value}")
print("\n── Audit trail:")
for entry in agent1.state_log:
    print(f"   {entry}")

## Step 7 — Task 2: Multi-Step (Two Tool Calls)

A task that requires calculate AND convert_currency.  
Watch the FSM loop back through `REFLECTING → PLANNING → CALLING → OBSERVING → REFLECTING → DONE`.

In [ ]:
agent2 = FSMAgent(tools=TOOLS, tool_executor=tool_executor, max_turns=12)
result2 = agent2.run(
    task="A project costs $85,000 per year. What is the monthly cost in GBP and EUR?",
    system_prompt=SYSTEM
)
print(f"\n── Answer: {result2}")
print(f"\n── Final state: {agent2.state.value}")
print("\n── Audit trail:")
for entry in agent2.state_log:
    print(f"   {entry}")

## Step 8 — Task 3: Budget Guard Demo

We set `max_turns=2` on a complex task. The budget guard fires.  
Watch the state log — it won't reach `DONE`.

This is how you prevent runaway agents in production.

In [ ]:
agent3 = FSMAgent(tools=TOOLS, tool_executor=tool_executor, max_turns=2)
result3 = agent3.run(
    task="Calculate compound interest on £10,000 at 5% for 10 years, then convert to USD, EUR, JPY, and INR.",
    system_prompt=SYSTEM
)
print(f"\n── Final state: {agent3.state.value}")
print("\n── Audit trail (notice where it stopped):")
for entry in agent3.state_log:
    print(f"   {entry}")

## Key Takeaways

| Point | Why it matters |
|-------|---------------|
| Explicit state makes bugs visible | Hidden state hides bugs until they corrupt production data |
| Guards fire BEFORE state changes | An illegal move raises an exception — caught in tests, not in prod |
| Budget guard is separate from state logic | It fires every turn regardless of state — no agent runs forever |
| The audit trail is free observability | In production: write `state_log` to CloudWatch. Compliance loves it. |
| Every framework does this | LangGraph, CrewAI, Bedrock Agents — all implement some version of this FSM |

---

**Next lab:** Lab 7 — Evaluator-Optimizer  
Two separate agents: one generates, one scores. Watch quality improve across iterations.